
# Kuroshio Daily R5.1 — QA 失败诊断及恒等残差资格实验

上一轮 R5 的 `qualified=false` 是资格门槛生效，不是 PyTorch 程序错误。

**先读取既有 R5 的逐样本 QA CSV 确认为什么失败。** 如果结果未上传到对话，但已经存在 WSL 工作目录，本 Notebook 可自动读取并生成逐年/逐提前期/逐 seed 差异。然后用严格小实验比较三种**最小机制**：

- **R51-IdentityTemporal**：保持原空间 TCN + LSTM，只加入恒等近似的真正时间卷积。修复此前 `LayerNorm(x + gate*conv(x))` 导致残差非恒等的问题。
- **R51-PersistenceDelta**：不加入时间卷积，仅让预测头学习当前 EOF 路径相对未来路径的增量 `PC_last + ΔPC`，避免强制衰减因子 0.45/0.30/0.18。
- **R51-Combined**：上述二者组合。

对照始终保留同参数 LSTM 与原 EOF-TCN-LSTM。先用 QA 2008/2016 诊断筛选，通过才在 2012/2020 做额外确认。所有结果均保存，失败时**正常结束**，不触发正式长训。

⚠️ 2008/2016 已经用于上一轮 R5 方案筛选，所以它们只能作为**研发验证**，不是全新独立验证。2012/2020 亦与历史实验设计存在时间关系，不能宣称真正盲测。正式论文的性能论证仍须如实披露全部多年滚动后报和基线表现。

**R5.1 不会自动启动正式长周期训练，不会改低 Gate，不会隐藏失败结果。**


In [ ]:
# ===== 0. 配置：先核实三个目录；改好后从头顺序运行 =====
from dataclasses import dataclass, asdict
from pathlib import Path
import os, json, math, gc, time, random, hashlib, warnings, sys
import numpy as np
import pandas as pd

@dataclass
class CFG:
    ROOT: str = '/mnt/d/KuroshioDaily1993_2025'
    DATASET_REL: str = 'final/dataset/daily_dual_branch_paths_1993_2025_FINAL.npz'
    HANDOFF_REL: str = 'final/reports/LONG_DAILY_1993_2025_HANDOFF.json'
    QC_INDEX_REL: str = 'final/reports/最终逐日路径索引.csv'
    MANUAL_QUEUE_REL: str = 'reports/07_人工复核队列.csv'
    MANUAL_METADATA_REL: str = 'manual_qc/metadata'
    OUTPUT_REL: str = 'R5_1_恒等残差资格实验'
    HISTORY: int = 180
    LEADS: tuple = (30, 60, 90)
    TEST_YEARS: tuple = (2005, 2009, 2013, 2017, 2021, 2024)
    PURGE_DAYS: int = 7
    MAX_INPUT_GAP: int = 2
    MAX_MISSING_FRAC: float = 0.03
    MIN_DATA_YEARS: int = 25
    MIN_ANNUAL_COVERAGE: float = 0.85
    MIN_TRAIN: int = 500
    MIN_VAL: int = 100
    MIN_TEST: int = 100
    DROP_BRANCH1_FIRST: int = 12  # 43->31; same old representation
    BRANCH2_MAX_LON: float = 145.0
    EOF_VARIANCE_TARGET: float = 0.99
    BATCH_SIZE: int = 32
    EPOCHS_B1: int = 70
    EPOCHS_B2: int = 70
    PATIENCE: int = 12
    LR: float = 0.001
    WEIGHT_DECAY: float = 0.0001
    DROPOUT: float = 0.30
    SEEDS: tuple = (20,21,22,23,24)
    AR_LAGS: tuple = (3, 7)
    AR_ALPHAS: tuple = (0.1, 1., 10.)
    CLIM_HALF_WINDOW_DAYS: int = 15
    CLIM_MIN_SAMPLES: int = 10
    BOOT_N: int = 1000
    BOOT_BLOCK_DAYS: int = 30
    FORCE_TRAIN: bool = False
    USE_AMP: bool = True
    RUN_FULL_TRAINING: bool = False   # 必须先通过 Cell 5 和 Cell 9 的资格Gate
    TRAIN_OTHER_HYBRIDS: bool = False # 下一轮网络结构比较可启用，不改baseline主表
    FIG_DPI: int = 300
    PILOT_TEST_YEARS: tuple = (2009, 2017)  # QA: 2008/2016, inner early-stop: 2007/2015
    PILOT_SEEDS: tuple = (20, 21, 22)
    PILOT_EPOCHS: int = 24
    PILOT_PATIENCE: int = 7
    GATE_MIN_GAIN_KM: float = 0.50
    GATE_MAX_LEAD_REGRESSION_KM: float = 2.0
    ALLOW_UNQUALIFIED_FULL: bool = False  # 固定实验合同，失败则只出诊断，不进行正式长训
    CONFIRM_YEARS: tuple = (2006,2010,2014,2018,2022,2023)
    PHYS_LOSS_WEIGHT: float = 0.35
    PHYS_KM_SCALE: float = 50.0
    TEMPORAL_DILATIONS: tuple = (1, 7, 21)
    TRAIN_ORIGINAL_ETL: bool = True
    QUALIFY_ON_INNER_QA: bool = True
    CONFIRM_PILOT_YEARS: tuple = (2013, 2021)  # additional QA 2012 and 2020
    IDENTITY_GATE_INIT: float = -4.0
    RUN_CONFIRMATION: bool = True

cfg=CFG(); ROOT=Path(cfg.ROOT); DS=ROOT/cfg.DATASET_REL; OUT=ROOT/cfg.OUTPUT_REL
DATA=OUT/'实验数据'; CKPT=OUT/'模型权重'; PRED=OUT/'逐样本预测'; REP=OUT/'报告'; FZH=OUT/'图表'/'中文'; FEN=OUT/'图表'/'英文'
for d in (DATA,CKPT,PRED,REP,FZH,FEN): d.mkdir(parents=True,exist_ok=True)
print(json.dumps(asdict(cfg),indent=2,ensure_ascii=False))
if not DS.exists(): raise FileNotFoundError(f'未发现真正的冻结数据：{DS}。请先运行人工QC notebook最后的冻结与数据构建单元。')

In [ ]:
# ===== 1. GPU与字体/依赖检查（不合格立即停止，避免白训练）=====
import importlib.util
required=('torch','scipy','sklearn','matplotlib','tqdm','pandas','numpy')
missing=[v for v in required if importlib.util.find_spec(v) is None]
if missing: raise RuntimeError('WSL环境缺依赖：'+', '.join(missing)+'；用当前Kernel对应python执行 python -m pip install torch scipy scikit-learn matplotlib tqdm pandas numpy')
import torch, torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from scipy.spatial import cKDTree
from sklearn.linear_model import Ridge
import matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.font_manager import FontProperties
from tqdm.auto import tqdm
print('kernel=',sys.executable,'torch=',torch.__version__,'CUDA=',torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('本轮RTX5080正式训练要求CUDA GPU；请检查WSL nvidia-smi 与 PyTorch CUDA构建。')
print('GPU=',torch.cuda.get_device_name(0)); DEV=torch.device('cuda')
torch.set_float32_matmul_precision('high')
font_paths=['/mnt/c/Windows/Fonts/msyh.ttc','/mnt/c/Windows/Fonts/simhei.ttf','/mnt/c/Windows/Fonts/simsun.ttc','/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc']
CJK_NAME=None
for path in font_paths:
    if Path(path).exists():
        try:
            font_manager.fontManager.addfont(path)
            CJK_NAME=FontProperties(fname=path).get_name(); break
        except Exception: pass
if CJK_NAME is None:
    raise RuntimeError('中文字体缺失：请在WSL安装 sudo apt install fonts-noto-cjk 或检查 Windows /mnt/c/Windows/Fonts，避免绘图乱码。')
print('Chinese font=',CJK_NAME)

In [ ]:

# ===== 1A. 先读取旧R5的真实QA诊断文件（没有时不阻断后续实验） =====
OLD_DIR=ROOT/'R5_轻量时序残差优化'/'实验数据'
old_qa_file=OLD_DIR/'R5_03_机制资格QA逐样本误差.csv'
old_gate_file=OLD_DIR/'R5_05_机制资格硬门槛.csv'
print('旧QA逐样本存在:',old_qa_file.exists())
print('旧Gate逐项存在:',old_gate_file.exists())

if old_gate_file.exists():
    old_gate=pd.read_csv(old_gate_file)
    display(old_gate)
    old_gate.to_csv(DATA/'R51_00_旧R5逐候选资格复核.csv',index=False,encoding='utf-8-sig')
else:
    print('本地缺少旧R5_05 CSV，当前上传的JSON只有合格与否，不能推断各模型的真实指标。')

if old_qa_file.exists():
    old_qa=pd.read_csv(old_qa_file)
    required=['QA年份','模型','种子','提前期_天','横向路径误差MSE_km2']
    if not set(required).issubset(old_qa.columns):
        raise RuntimeError(f'旧QA CSV缺少字段: {set(required)-set(old_qa.columns)}')
    old_runs=(old_qa.groupby(['QA年份','模型','种子','提前期_天'],as_index=False)
         .agg(RMSE_km=('横向路径误差MSE_km2',lambda x:float(np.sqrt(np.mean(x))))))
    old_yr=(old_runs.groupby(['QA年份','模型','提前期_天'],as_index=False)['RMSE_km'].mean())
    old_mean=(old_yr.groupby(['模型','提前期_天'],as_index=False)['RMSE_km'].mean())
    old_mean.to_csv(DATA/'R51_00_旧R5各模型QA提前期误差.csv',index=False,encoding='utf-8-sig')
    old_yr.to_csv(DATA/'R51_00_旧R5逐年提前期误差.csv',index=False,encoding='utf-8-sig')
    old_runs.to_csv(DATA/'R51_00_旧R5逐种子提前期误差.csv',index=False,encoding='utf-8-sig')
    mat=old_mean.pivot(index='模型',columns='提前期_天',values='RMSE_km')
    display(mat)
    if 'Matched-LSTM' in mat.index and 'EOF-TCN-LSTM' in mat.index:
        qa_delta=mat.sub(mat.loc['Matched-LSTM'],axis=1)
        qa_delta.to_csv(DATA/'R51_00_旧R5相对MatchedLSTM误差差.csv',encoding='utf-8-sig')
        display(qa_delta)
else:
    print('未找到旧QA逐样本CSV；请提供 R5_03_机制资格QA逐样本误差.csv 才能对旧机制做定量失败归因。')


In [ ]:
# ===== 2. 最终数据读取、年度覆盖、人工QC冻结核对 =====
with np.load(DS,allow_pickle=False) as z:
    dates=pd.DatetimeIndex(pd.to_datetime(z['dates'])).normalize()
    b1=np.asarray(z['branch1_lon'],np.float32)
    lat1=np.asarray(z['branch1_target_lats'],float)
    b2=np.asarray(z['branch2_lat'],np.float32)
    lon2=np.asarray(z['branch2_target_lons'],float)
    source=np.asarray(z['source']).astype(str) if 'source' in z.files else np.array(['unknown']*len(dates))
if not (len(dates)==len(b1)==len(b2)==len(source)): raise RuntimeError('dates / 两分支 / source 数量不一致')
if dates.has_duplicates: raise RuntimeError('发现重复日期，禁止将其直接用于训练')
ord=np.argsort(dates.values); dates=dates[ord]; b1=b1[ord];b2=b2[ord];source=source[ord]
if not np.isfinite(b1).all() or not np.isfinite(b2).all(): raise RuntimeError('冻结数据中含 NaN/Inf')

# 物理范围异常只作质控警示（不随意删除或修改路径），便于发现断面插值外推造成的不可信日
suspicious=(~((b1>=119)&(b1<=140)).all(axis=1)) | (~((b2>=15)&(b2<=50)).all(axis=1))
if suspicious.any():
    suspect_df=pd.DataFrame({'异常日期':dates[suspicious], '原因':'双分支坐标超出宽松物理范围，需人工核对'})
    suspect_df.to_csv(DATA/'01_待人工复查的物理范围异常路径.csv',index=False,encoding='utf-8-sig')
    warnings.warn(f'发现 {suspicious.sum()} 天路径越过宽松物理边界，建议正式训练前核实此文件')

if b1.shape[1]<=cfg.DROP_BRANCH1_FIRST: raise RuntimeError('分支1维数过小')
b1=b1[:,cfg.DROP_BRANCH1_FIRST:];lat1=lat1[cfg.DROP_BRANCH1_FIRST:]
keep2=lon2<=cfg.BRANCH2_MAX_LON+1e-8;b2=b2[:,keep2];lon2=lon2[keep2]
if len(b1)!=len(b2) or len(lat1)!=b1.shape[1] or len(lon2)!=b2.shape[1]: raise RuntimeError('双分支物理坐标不匹配')
if dates.min().year>1993 or dates.max().year<2024: raise RuntimeError('数据不覆盖1993–2024所需区间，不可宣称多年全历史验证')
cal=pd.date_range(dates.min(),dates.max(),freq='D'); pos=cal.get_indexer(dates)
valid=np.zeros(len(cal),bool);valid[pos]=True
A1=np.full((len(cal),b1.shape[1]),np.nan,np.float32);A2=np.full((len(cal),b2.shape[1]),np.nan,np.float32)
A1[pos]=b1; A2[pos]=b2
# causal forward fill (ONLY history, never target). All targets later read A1/A2 raw by exact date.
def causal_fill(raw,limit): return pd.DataFrame(raw).ffill(limit=limit).to_numpy(dtype=np.float32)
F1=causal_fill(A1,cfg.MAX_INPUT_GAP);F2=causal_fill(A2,cfg.MAX_INPUT_GAP)
annual=pd.DataFrame({'日期':cal,'有效':valid});annual['年份']=annual['日期'].dt.year
cover=annual.groupby('年份',as_index=False).agg(理论天数=('日期','size'),有效天数=('有效','sum'))
cover['有效比例']=cover['有效天数']/cover['理论天数'];cover.to_csv(DATA/'01_逐年覆盖率.csv',index=False,encoding='utf-8-sig')
if (cover['有效比例']>=cfg.MIN_ANNUAL_COVERAGE).sum()<cfg.MIN_DATA_YEARS:
    raise RuntimeError('具备≥85%有效覆盖的年份不足25年，请优先排查未完成人工修复或缺失年；详情见01_逐年覆盖率.csv')
hand=ROOT/cfg.HANDOFF_REL
if hand.exists():
    ho=json.loads(hand.read_text(encoding='utf-8'))
    print('HANDOFF final_daily_paths=',ho.get('final_daily_paths'))
    if int(ho.get('final_daily_paths',-1))!=len(dates): raise RuntimeError('HANDOFF与冻结npz样本数不一致')
else: warnings.warn('未发现HANDOFF.json；冻结数据可供资格检查，但交接记录尚未核实')
idxp=ROOT/cfg.QC_INDEX_REL
if idxp.exists():
    idx=pd.read_csv(idxp)
    if 'status' in idx.columns:
        unresolved=idx[~idx['status'].isin(['final','missing_raw','unusable'])]
        if len(unresolved): raise RuntimeError(f'最终逐日索引仍有 {len(unresolved)} 条未处理状态，停止训练')
queue=ROOT/cfg.MANUAL_QUEUE_REL; mdir=ROOT/cfg.MANUAL_METADATA_REL
if queue.exists():
    qq=pd.read_csv(queue)
    qdates=pd.to_datetime(qq['date']).dt.strftime('%Y%m%d') if len(qq) else []
    missing_meta=[q for q in qdates if not (mdir/f'{q}.json').exists()]
    if missing_meta: raise RuntimeError(f'{len(missing_meta)} 个manual复核队列日期尚无处理metadata: {missing_meta[:10]}')
    bad_meta=[]
    allowed={'manual_corrected','accepted_auto_after_manual_qc','unusable'}
    for q in qdates:
        try:
            state=json.loads((mdir/f'{q}.json').read_text(encoding='utf-8')).get('status')
            if state not in allowed:bad_meta.append((q,state))
        except Exception as e:bad_meta.append((q,repr(e)))
    if bad_meta:raise RuntimeError(f'{len(bad_meta)} 条人工QC仍不是已确认状态: {bad_meta[:10]}')
# CMEMS来源切换边界连续性检查：仅警告，不把真实大蛇行直接当成数据损坏
edge=pd.Timestamp('2025-01-01')
prev=np.flatnonzero(dates<edge)
post=np.flatnonzero(dates>=edge)
if len(prev) and len(post):
    i,j=int(prev[-1]),int(post[0])
    if (dates[j]-dates[i]).days<=5:
        d1=float(np.sqrt(np.mean((b1[j]-b1[i])**2)))
        d2=float(np.sqrt(np.mean((b2[j]-b2[i])**2)))
        pd.DataFrame([{'产品交界前':dates[i],'交界后':dates[j],
                       '分支1逐日经度RMSE_度':d1,'分支2逐日纬度RMSE_度':d2}]).to_csv(
            DATA/'01_2024到2025产品交界连续性.csv',index=False,encoding='utf-8-sig')
        if d1>1.0 or d2>1.5:
            warnings.warn('2024-2025 CMEMS来源交界发生较大路径跳变，应先检查跨产品物理一致性与流轴QC')
print('DATA PASS:',len(dates),'days;',dates.min().date(),'to',dates.max().date(),'; B1',b1.shape[1],'; B2',b2.shape[1])
display(cover)

In [ ]:
# ===== 3. 创建样本索引，不展开 [样本,180,特征] 巨型数组 =====
# target index = issue+lead; test targets require true valid day. input missing <=2 continuous days.
L=max(cfg.LEADS); H=cfg.HISTORY
missing=(~valid).astype(np.int32)
issue_indices=[];audit=[]
for i in range(H-1,len(cal)-L):
    if not valid[i]: continue
    win=missing[i-H+1:i+1]
    if win.mean()>cfg.MAX_MISSING_FRAC: continue
    if len(win) and np.max(np.diff(np.flatnonzero(np.r_[1,win==0,1]))-1)>cfg.MAX_INPUT_GAP: continue
    if any(not valid[i+d] for d in cfg.LEADS): continue
    if not (np.isfinite(F1[i-H+1:i+1]).all() and np.isfinite(F2[i-H+1:i+1]).all()): continue
    issue_indices.append(i)
ISS=np.asarray(issue_indices,np.int32)
if len(ISS)==0: raise RuntimeError('没有有效180→30/60/90 anchors，需检查缺日与冻结数据')
anchor=pd.DataFrame({'起报日期':cal[ISS]})
for l in cfg.LEADS: anchor[f'{l}天目标日期']=cal[ISS+l]
anchor.to_csv(DATA/'02_全部有效预测起报日期.csv',index=False,encoding='utf-8-sig')
print('anchor count=',len(ISS),' first=',cal[ISS[0]],'last=',cal[ISS[-1]])

In [ ]:
# ===== 4. 真正的 expanding-window 多年后报：test year = 2005/2009/2013/2017/2021/2024 =====
FOLDS=[];fr=[]; splitrows=[]
for fold_id,year in enumerate(cfg.TEST_YEARS,1):
    valstart=pd.Timestamp(year-1,1,1); teststart=pd.Timestamp(year,1,1)
    testend=pd.Timestamp(year,12,31)
    train_cut=valstart-pd.Timedelta(days=cfg.PURGE_DAYS)
    val_cut=teststart-pd.Timedelta(days=cfg.PURGE_DAYS)
    issues=cal[ISS]; latest=cal[ISS+L]
    tr=ISS[(latest<train_cut)]
    va=ISS[(issues>=valstart)&(issues<teststart)&(latest<val_cut)]
    te=ISS[(issues>=teststart)&(issues<=testend)]
    if len(tr)<cfg.MIN_TRAIN or len(va)<cfg.MIN_VAL or len(te)<cfg.MIN_TEST:
        raise RuntimeError(f'Fold {year} 样本不足: tr={len(tr)} val={len(va)} test={len(te)}；请查该年的日路径覆盖和连续缺日')
    if not(cal[tr+L].max()<cal[va].min()-pd.Timedelta(days=cfg.PURGE_DAYS) and cal[va+L].max()<cal[te].min()-pd.Timedelta(days=cfg.PURGE_DAYS)):
        raise RuntimeError(f'Fold {year} 发现严格时序泄漏')
    if (len(set(cal[tr+L])&set(cal[va+L])) or len(set(cal[tr+L])&set(cal[te+L])) or len(set(cal[va+L])&set(cal[te+L]))):
        raise RuntimeError(f'Fold {year} label date overlap')
    ff={'fold':fold_id,'year':year,'train':tr,'val':va,'test':te,'train_cut':train_cut,'val_cut':val_cut}
    FOLDS.append(ff)
    fr.append({'折次':fold_id,'测试年份':year,'训练样本':len(tr),'验证样本':len(va),'测试样本':len(te),
               '训练最晚标签日期':cal[tr+L].max(),'验证最早起报日期':cal[va].min(),
               '验证最晚标签日期':cal[va+L].max(),'测试最早起报日期':cal[te].min(),'测试最晚标签日期':cal[te+L].max(),'因果审计':'PASS'})
    for label,ix in [('训练',tr),('验证',va),('测试',te)]:
        splitrows.extend({'折次':fold_id,'测试年份':year,'集合':label,'起报日期':str(cal[i].date()),'最晚目标日期':str(cal[i+L].date())} for i in ix)
foldtab=pd.DataFrame(fr); foldtab.to_csv(DATA/'03_多年滚动后报折次及防泄漏审计.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(splitrows).to_csv(DATA/'03_逐样本时序划分.csv',index=False,encoding='utf-8-sig')
print('[STRICT MULTI-YEAR CAUSAL HINDCAST] ALL PASS'); display(foldtab)

In [ ]:
# ===== 5. 每折 train-only 标准化 + 自适应 99% EOF =====
# 使用该fold训练期的独立日记录，而不是重叠180d窗口重复叠加做SVD；内存恒定在 O(feature²)
class DailyEOF:
    def fit(self,obs,target=0.99):
        obs=np.asarray(obs,np.float64)
        self.mu=np.mean(obs,axis=0); self.sd=np.std(obs,axis=0); self.sd[self.sd<1e-6]=1.
        z=(obs-self.mu)/self.sd
        cov=(z.T@z)/max(1,len(z)-1)
        eig,vec=np.linalg.eigh(cov)
        idx=np.argsort(eig)[::-1];eig=np.maximum(eig[idx],0);vec=vec[:,idx]
        rat=np.cumsum(eig)/(eig.sum()+1e-12); n=int(np.searchsorted(rat,target)+1)
        self.V=np.asarray(vec[:,:n],np.float32);self.explained=float(rat[n-1]);self.n=n
        return self
    def pc(self,x): return (((np.asarray(x)-self.mu)/self.sd)@self.V).astype(np.float32)
    def invert(self,pc): return ((np.asarray(pc)@self.V.T)*self.sd+self.mu).astype(np.float32)
    def z(self,x): return ((np.asarray(x)-self.mu)/self.sd).astype(np.float32)

def make_fold_data(f):
    train_days=(cal< f['train_cut']) & valid
    pp1=DailyEOF().fit(A1[train_days],cfg.EOF_VARIANCE_TARGET)
    pp2=DailyEOF().fit(A2[train_days],cfg.EOF_VARIANCE_TARGET)
    for pp in (pp1,pp2):
        if pp.explained<cfg.EOF_VARIANCE_TARGET-1e-6: raise RuntimeError('EOF 99%训练方差Gate未通过')
    # approx memory: ~17k days * 2*(31+277) floats; never expands 180 day windows
    z1=pp1.z(F1);z2=pp2.z(F2)
    pcs1=pp1.pc(F1);pcs2=pp2.pc(F2)
    ppath=CKPT/f'第{f["fold"]}折_{f["year"]}年'; ppath.mkdir(exist_ok=True,parents=True)
    np.savez_compressed(ppath/'训练期预处理参数.npz',
        b1_mu=pp1.mu,b1_sd=pp1.sd,b1_V=pp1.V,b2_mu=pp2.mu,b2_sd=pp2.sd,b2_V=pp2.V,
        branch1_lat=lat1,branch2_lon=lon2,train_label_cutoff=str(f['train_cut'].date()))
    return {'pp1':pp1,'pp2':pp2,'z1':z1,'z2':z2,'p1':pcs1,'p2':pcs2,'path':ppath}

# 只预审一折，防止全局重复保留大数组
_testFD=make_fold_data(FOLDS[0]);print('fold1 EOF modes:',_testFD['pp1'].n,_testFD['pp2'].n,'explained:',_testFD['pp1'].explained,_testFD['pp2'].explained)
del _testFD;gc.collect()
print('[TRAIN-ONLY EOF 99%] PASS')

In [ ]:
# ===== 6. 动态加载数据集，训练时每个batch只展开当前180天窗口 =====
class CausalWindowDataset(Dataset):
    def __init__(self,issues,scaled,pcs):
        self.issues=np.asarray(issues,np.int32);self.scaled=scaled;self.pcs=pcs
    def __len__(self): return len(self.issues)
    def __getitem__(self,j):
        i=int(self.issues[j]);z=self.scaled[i-H+1:i+1]
        dz=np.empty_like(z);dz[0]=0;dz[1:]=z[1:]-z[:-1]
        x=np.concatenate([z,dz],axis=-1)
        y=np.stack([self.pcs[i+d] for d in cfg.LEADS])
        return torch.from_numpy(x.copy()),torch.from_numpy(y.copy())

class StepEncoder(nn.Module):
    # 与旧原稿的per-step TCN/Conv编码保持一致：输入路径特征卷积，然后LSTM承担时间建模
    def __init__(self,feature,branch,kind='tcn'):
        super().__init__();self.kind=kind;self.drop=nn.Dropout(cfg.DROPOUT)
        if kind=='tcn':
            channels=16 if branch==1 else 32;kernel=3 if branch==1 else 5
            self.conv=nn.Conv1d(1,channels,kernel,dilation=2)
            self.bn=nn.BatchNorm1d(channels)
            conv_len=feature+2-2*(kernel-1)
            self.fc=nn.Linear(channels*conv_len,32 if branch==1 else 64)
            self.out_dim=self.fc.out_features
        elif kind=='conv':
            c1=16 if branch==1 else 32;c2=32 if branch==1 else 64
            self.c1=nn.Conv1d(1,c1,3,padding=1);self.c2=nn.Conv1d(c1,c2,3,padding=1)
            self.out_dim=c2*feature
        else: raise ValueError(kind)
    def forward(self,x):
        B,T,F=x.shape; z=x.reshape(B*T,1,F)
        if self.kind=='tcn':
            z=torch.nn.functional.pad(z,(2,0)) # 与R3原架构一致的左侧填充
            z=self.drop(torch.relu(self.bn(self.conv(z))))
            z=self.drop(torch.relu(self.fc(z.flatten(1))))
        else:
            z=self.drop(torch.relu(self.c1(z)));z=self.drop(torch.relu(self.c2(z)));z=z.flatten(1)
        return z.reshape(B,T,-1)

class Hybrid(nn.Module):
    def __init__(self,feature,pc_dim,branch,kind='tcn',recur='lstm'):
        super().__init__();self.enc=StepEncoder(feature,branch,kind)
        h=32 if branch==1 else 64
        R=nn.GRU if recur=='gru' else nn.LSTM
        self.rnn=R(self.enc.out_dim,h,batch_first=True)
        self.fc=nn.Linear(h,3*pc_dim);self.pc_dim=pc_dim
    def forward(self,x):
        h,_=self.rnn(self.enc(x));return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

class MatchedLSTM(nn.Module):
    def __init__(self,feature,pc_dim,hidden):
        super().__init__();self.rnn=nn.LSTM(feature,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pc_dim);self.pc_dim=pc_dim
    def forward(self,x):
        h,_=self.rnn(x);return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

def nparams(m): return sum(p.numel() for p in m.parameters())
def match_lstm(feature,pc_dim,branch):
    target=nparams(Hybrid(feature,pc_dim,branch))
    # Pytorch single-layer LSTM: 4h(feature+h+2) + 3pc*h + 3pc
    opts=range(8,1025)
    h=min(opts,key=lambda hh:abs(4*hh*(feature+hh+2)+(3*pc_dim)*(hh+1)-target))
    actual=4*h*(feature+h+2)+(3*pc_dim)*(h+1)
    if abs(actual-target)/target>0.08: raise RuntimeError('Matched-LSTM参数量相对差>8%，不满足公平基线')
    return h,target,actual

def make_net(name,branch,fd):
    feature=fd[f'z{branch}'].shape[1]*2;pdim=fd[f'pp{branch}'].n
    if name=='Matched-LSTM':
        h,_,_=match_lstm(feature,pdim,branch);return MatchedLSTM(feature,pdim,h)
    if name=='EOF-TCN-LSTM': return Hybrid(feature,pdim,branch,'tcn','lstm')
    if name=='EOF-Conv-LSTM': return Hybrid(feature,pdim,branch,'conv','lstm')
    if name=='EOF-Conv-GRU': return Hybrid(feature,pdim,branch,'conv','gru')
    if name=='EOF-TCN-GRU': return Hybrid(feature,pdim,branch,'tcn','gru')
    raise ValueError(name)


# ===== R5.1: 真正可控的近恒等残差时序模块 =====
class IdentityTemporalResidual(nn.Module):
    def __init__(self, dim, dilations=(1,7,21), dropout=0.10):
        super().__init__()
        self.norms=nn.ModuleList([nn.LayerNorm(dim) for _ in dilations])
        self.blocks=nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(dim,dim,3,dilation=d,groups=dim,bias=False),
                nn.Conv1d(dim,dim,1),nn.GELU(),nn.Dropout(dropout)
            ) for d in dilations
        ])
        self.gates=nn.ParameterList([
            nn.Parameter(torch.tensor(float(cfg.IDENTITY_GATE_INIT))) for _ in dilations
        ])
        self.dilations=tuple(int(d) for d in dilations)

    def forward(self,x):
        for norm,block,g,d in zip(self.norms,self.blocks,self.gates,self.dilations):
            # 修复：先LN作用在扰动分支，主路径保留原始幅度。
            # 卷积单元仅用 <=t 的特征，不访问未来输入。
            inp=norm(x).transpose(1,2)
            z=block(torch.nn.functional.pad(inp,(2*d,0))).transpose(1,2)
            x=x+torch.sigmoid(g)*z
        return x

class R51Hybrid(nn.Module):
    def __init__(self,feature,pc_dim,branch,name,projection):
        super().__init__()
        self.name=name
        self.raw_features=feature//2
        self.encoder=StepEncoder(feature,branch,'tcn')
        use_temporal=name in ('R51-IdentityTemporal','R51-Combined')
        self.temporal=(IdentityTemporalResidual(self.encoder.out_dim,cfg.TEMPORAL_DILATIONS)
                       if use_temporal else nn.Identity())
        hidden=32 if branch==1 else 64
        self.rnn=nn.LSTM(self.encoder.out_dim,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pc_dim)
        self.pc_dim=pc_dim
        self.use_delta=name in ('R51-PersistenceDelta','R51-Combined')
        self.register_buffer('proj_V',torch.as_tensor(projection,dtype=torch.float32))
        if self.use_delta:
            nn.init.zeros_(self.fc.weight)
            nn.init.zeros_(self.fc.bias)

    def forward(self,x):
        z=self.temporal(self.encoder(x))
        h,_=self.rnn(z)
        delta=self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)
        if self.use_delta:
            # 精确 persistence 初始化，不强加固定 mean-reversion 衰减。
            pc_last=x[:,-1,:self.raw_features]@self.proj_V
            return pc_last[:,None,:]+delta
        return delta

R51_NAMES=('R51-IdentityTemporal','R51-PersistenceDelta','R51-Combined')

def make_r5_model(name,branch,fd):
    feat=fd[f'z{branch}'].shape[1]*2
    pdim=fd[f'pp{branch}'].n
    if name in R51_NAMES:
        return R51Hybrid(feat,pdim,branch,name,fd[f'pp{branch}'].V)
    if name in ('EOF-TCN-LSTM','Matched-LSTM'):
        return make_net(name,branch,fd)
    raise ValueError(name)

def model_loss(pred,target,fd,branch,name):
    # 诊断固定：全部使用与原R4一致的PC-MSE，暂不引入未验证物理损失，
    # 这样可明确归因于“时序残差”或“状态增量头”的结构变化。
    return torch.nn.functional.mse_loss(pred,target)


In [ ]:
# ===== 7. 五种基线/评估的科学定义（季节态和递推AR-EOF仅使用训练日期）=====
def persistence_pred(ix):
    return np.repeat(A1[ix,None,:],3,axis=1).astype(np.float32),np.repeat(A2[ix,None,:],3,axis=1).astype(np.float32)

def climatology_model(raw,trainmask):
    ids=np.flatnonzero(trainmask);dates_train=cal[ids]
    doy=np.array([t.dayofyear for t in dates_train],int)
    clims=[];glob=np.mean(raw[ids],axis=0)
    for day in range(1,367):
        cyc=np.minimum(np.abs(doy-day),366-np.abs(doy-day))
        ids2=ids[cyc<=cfg.CLIM_HALF_WINDOW_DAYS]
        clims.append(np.mean(raw[ids2],axis=0) if len(ids2)>=cfg.CLIM_MIN_SAMPLES else glob)
    return np.asarray(clims,np.float32)

def fit_recursive_AR(pc,rawvalid,train_cut,p,alpha):
    end=cal.searchsorted(train_cut,side='left')
    # 不把缺失日上已有causal fill的路径当成独立观测标签
    t=np.arange(p,end)
    ok=rawvalid[t].copy()
    for lag in range(1,p+1): ok &= rawvalid[t-lag]
    t=t[ok]
    if len(t)<200: raise RuntimeError('AR训练可用完整逐日连续片段不足200')
    XX=np.stack([pc[t-i] for i in range(p,0,-1)],axis=1).reshape(len(t),-1)
    yy=pc[t]
    reg=Ridge(alpha=alpha).fit(XX,yy)
    train_sd=np.std(pc[:end][rawvalid[:end]],axis=0)
    clip=5*np.maximum(train_sd,0.5)
    return reg,clip

def recursive_forecast(reg,clip,pc,issue,p):
    hist=np.asarray(pc[issue-p+1:issue+1],np.float32)
    if len(hist)!=p or not np.isfinite(hist).all(): raise RuntimeError('AR history bad')
    outs=[]
    for t in range(1,91):
        nxt=reg.predict(hist.reshape(1,-1))[0]
        nxt=np.clip(nxt,-clip,clip).astype(np.float32)
        hist=np.vstack((hist[1:],nxt))
        if t in cfg.LEADS: outs.append(nxt.copy())
    return np.stack(outs)

def tune_AR(fd,f,branch):
    pc=fd[f'p{branch}']; pp=fd[f'pp{branch}']; train_cut=f['train_cut']
    best=None
    validx=f['val'][::max(1,len(f['val'])//70)]
    for p in cfg.AR_LAGS:
        for alpha in cfg.AR_ALPHAS:
            reg,clip=fit_recursive_AR(pc,valid,train_cut,p,alpha)
            pred=np.stack([recursive_forecast(reg,clip,pc,int(i),p) for i in validx]);truth=np.stack([[pc[i+d] for d in cfg.LEADS] for i in validx])
            err=float(np.mean((pred-truth)**2))
            if best is None or err<best['val_mse']: best={'p':p,'alpha':alpha,'reg':reg,'clip':clip,'val_mse':err}
    return best

In [ ]:
# ===== 8. 物理公里误差：局地投影+最近点相邻线段的双向正交路径距离 =====
# 对于仅122-145E的分支2，重建得到共同定义域；不是对每一点简单经纬度平方误差。
# 下方是双向point-to-polyline近似cross-track，使用KDTree近邻三顶点及其邻接segments。
def merge_path(branch1,branch2):
    a=np.asarray(branch1,float);b=np.asarray(branch2,float)
    candidates=np.flatnonzero(lon2>a[-1]);
    k=int(candidates[np.argmin((lon2[candidates]-a[-1])**2+(b[candidates]-lat1[-1])**2)]) if len(candidates) else len(lon2)
    xs=np.r_[a,lon2[k:]];ys=np.r_[lat1,b[k:]]
    good=np.isfinite(xs)&np.isfinite(ys)
    return xs[good],ys[good]

def one_direction(A,B):
    # A,B: 2d coordinates (km); approximate nearest polyline segment for each A point
    if len(A)<2 or len(B)<2: return np.array([np.nan])
    tree=cKDTree(B);_,knn=tree.query(A,k=min(3,len(B)))
    knn=np.asarray(knn);knn=knn[:,None] if knn.ndim==1 else knn
    ix=np.unique(np.clip(np.concatenate([knn.ravel()-1,knn.ravel()]),0,len(B)-2))
    # segment shortlist globally may grow, but polynomial per path length (~300) and is vectorized
    v=B[ix+1]-B[ix];w=A[:,None,:]-B[ix][None,:,:]
    den=np.maximum(np.sum(v*v,axis=1),1e-12)
    t=np.clip(np.sum(w*v[None,:,:],axis=2)/den[None,:],0,1)
    q=B[ix][None,:,:]+t[:,:,None]*v[None,:,:]
    d=np.sqrt(np.sum((A[:,None,:]-q)**2,axis=2))
    return d.min(axis=1)

def errors_for_pair(pred1,pred2,truth1,truth2):
    x,y=merge_path(pred1,pred2);xt,yt=merge_path(truth1,truth2)
    lat0=np.mean(np.r_[y,yt]);lon0=np.mean(np.r_[x,xt]);c=np.cos(np.deg2rad(lat0))
    A=np.column_stack([(x-lon0)*111.32*c,(y-lat0)*111.32]);B=np.column_stack([(xt-lon0)*111.32*c,(yt-lat0)*111.32])
    d=np.r_[one_direction(A,B),one_direction(B,A)]
    return float(np.mean(d**2)),float(np.mean(d)),float(np.percentile(d,90))

def record_errors(fold,model,seed,issues,p1,p2):
    out=[]
    for n,i in enumerate(issues):
        for k,lead in enumerate(cfg.LEADS):
            y1=A1[i+lead];y2=A2[i+lead]
            mse,mean,p90=errors_for_pair(p1[n,k],p2[n,k],y1,y2)
            out.append({'折次':fold['fold'],'测试年份':fold['year'],'模型':model,'种子':seed,'起报日期':str(cal[i].date()),'目标日期':str(cal[i+lead].date()),'提前期_天':int(lead),
                        '横向路径误差MSE_km2':mse,'横向路径平均误差_km':mean,'横向路径P90_km':p90,
                        '分支1经度RMSE_度':float(np.sqrt(np.mean((p1[n,k]-y1)**2))),
                        '分支2纬度RMSE_度':float(np.sqrt(np.mean((p2[n,k]-y2)**2)))})
    return pd.DataFrame(out)

In [ ]:
# ===== 9. R5 训练封装、物理评价、可恢复 checkpoints =====
from contextlib import nullcontext

def r5_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def run_predict(net, issues, fd, branch):
    ds=CausalWindowDataset(issues,fd[f'z{branch}'],fd[f'p{branch}'])
    ld=DataLoader(ds,batch_size=cfg.BATCH_SIZE,shuffle=False,num_workers=0)
    arr=[];net.eval()
    with torch.no_grad():
        for xx,_ in ld: arr.append(net(xx.to(DEV)).float().cpu().numpy())
    return np.concatenate(arr,axis=0)

def fit_r5_branch(split, fd, name, seed, branch, root_dir, max_epochs=None, patience=None):
    max_epochs=cfg.PILOT_EPOCHS if max_epochs is None else max_epochs
    patience=cfg.PILOT_PATIENCE if patience is None else patience
    folder=root_dir/name/f'seed_{seed}'
    folder.mkdir(parents=True,exist_ok=True)
    weight=folder/f'branch{branch}_best.pt'; historyfile=folder/f'branch{branch}_history.csv'
    latest=folder/f'branch{branch}_latest.pt'
    # best checkpoint 附带实验合同签名；仅同一fold/样本划分/网络版本可复用
    contract={'name':name,'seed':seed,'branch':branch,'tr_n':int(len(split['train'])),
              'val_n':int(len(split['val'])),'tr_last':int(split['train'][-1]),
              'val_first':int(split['val'][0]),'max_epochs':max_epochs,
              'phys_weight':float(cfg.PHYS_LOSS_WEIGHT),'version':'R5.1-identity-v1'}
    contract_id=hashlib.sha256(json.dumps(contract,sort_keys=True).encode()).hexdigest()
    r5_seed(seed)
    net=make_r5_model(name,branch,fd).to(DEV)
    if weight.exists() and historyfile.exists() and not cfg.FORCE_TRAIN:
        ck=torch.load(weight,map_location='cpu',weights_only=True)
        if ck.get('contract_id')==contract_id:
            net.load_state_dict(ck['state']);return net,pd.read_csv(historyfile)
    ds_tr=CausalWindowDataset(split['train'],fd[f'z{branch}'],fd[f'p{branch}'])
    ds_va=CausalWindowDataset(split['val'],fd[f'z{branch}'],fd[f'p{branch}'])
    ld_tr=DataLoader(ds_tr,batch_size=cfg.BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=True)
    ld_va=DataLoader(ds_va,batch_size=cfg.BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
    opt=torch.optim.AdamW(net.parameters(),lr=cfg.LR,weight_decay=cfg.WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,factor=0.5,patience=4)
    amp=bool(cfg.USE_AMP)
    scaler=torch.amp.GradScaler('cuda',enabled=amp)
    start_epoch=1;best=float('inf');bestep=0;bad=0;hist=[]
    if latest.exists() and not cfg.FORCE_TRAIN:
        try:
            ck=torch.load(latest,map_location='cpu',weights_only=True)
            if ck.get('contract_id')==contract_id:
                net.load_state_dict(ck['state']);opt.load_state_dict(ck['opt']);scheduler.load_state_dict(ck['scheduler']);scaler.load_state_dict(ck['scaler'])
                start_epoch=int(ck['epoch'])+1;best=float(ck['best']);bestep=int(ck['bestep']);bad=int(ck['bad']);hist=ck['hist']
                print('[RESUME]',folder,'epoch',start_epoch)
        except Exception as e:print('[resume invalid]',e)
    for ep in range(start_epoch,max_epochs+1):
        net.train();ts=tn=0
        for x,y in ld_tr:
            x=x.to(DEV,non_blocking=True);y=y.to(DEV,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',enabled=amp):
                pred=net(x)
                loss=model_loss(pred.float(),y.float(),fd,branch,name)
            if not torch.isfinite(loss):raise RuntimeError(f'{name} B{branch}: non-finite loss')
            scaler.scale(loss).backward();scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(),1.0)
            scaler.step(opt);scaler.update();ts+=float(loss.detach())*len(x);tn+=len(x)
        net.eval();vs=vn=0
        with torch.no_grad():
            for x,y in ld_va:
                x=x.to(DEV);y=y.to(DEV)
                with torch.autocast('cuda',enabled=amp):
                    p=net(x);loss=model_loss(p.float(),y.float(),fd,branch,name)
                vs+=float(loss.detach())*len(x);vn+=len(x)
        val=vs/max(1,vn);scheduler.step(val)
        hist.append({'epoch':ep,'train_loss':ts/max(1,tn),'val_loss':val,'lr':opt.param_groups[0]['lr']})
        if val<best-1e-7:
            best=val;bestep=ep;bad=0
            torch.save({'state':{k:v.detach().cpu() for k,v in net.state_dict().items()},'contract_id':contract_id,'epoch':ep,'best_val':best},weight)
        else:bad+=1
        if ep%5==0 or bad>=patience or ep==max_epochs:
            torch.save({'state':{k:v.detach().cpu() for k,v in net.state_dict().items()},'opt':opt.state_dict(),
                        'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'epoch':ep,'best':best,
                        'bestep':bestep,'bad':bad,'hist':hist,'contract_id':contract_id},latest)
        if ep==1 or ep%10==0 or bad>=patience:print(f'{name} B{branch} seed{seed} ep{ep} train={hist[-1]["train_loss"]:.5f} val={val:.5f}')
        if bad>=patience:break
    pd.DataFrame(hist).to_csv(historyfile,index=False,encoding='utf-8-sig')
    if latest.exists():latest.unlink()
    ck=torch.load(weight,map_location='cpu',weights_only=True)
    net.load_state_dict(ck['state']);net.eval()
    return net,pd.DataFrame(hist)

def evaluate_issues(f,name,seed,fd,issues,bmodel1,bmodel2):
    return record_errors(f,name,seed,issues,bmodel1,bmodel2)

def get_predictions(name,split,fd,seed,folder,max_epochs=None,patience=None):
    branches=[]
    for branch in (1,2):
        net,hist=fit_r5_branch(split,fd,name,seed,branch,folder,max_epochs,patience)
        pr=run_predict(net,split.get('qa',split.get('test')),fd,branch)
        branches.append(fd[f'pp{branch}'].invert(pr))
        del net,pr;torch.cuda.empty_cache()
    return branches

In [ ]:
# ===== 10. R5 架构 / 纯因果 / 梯度冒烟预检 =====
f=FOLDS[0];fd=make_fold_data(f)
smoke=[]
for name in ['Matched-LSTM','EOF-TCN-LSTM','R51-IdentityTemporal','R51-PersistenceDelta','R51-Combined']:
    for b in (1,2):
        m=make_r5_model(name,b,fd).to(DEV)
        ds=CausalWindowDataset(f['train'][:12],fd[f'z{b}'],fd[f'p{b}'])
        x,y=next(iter(DataLoader(ds,batch_size=3)))
        x=x.to(DEV);y=y.to(DEV)
        m.eval()
        with torch.no_grad():
            p=m(x)
            if tuple(p.shape)!=tuple(y.shape):raise RuntimeError(f'{name} shape mismatch {tuple(p.shape)} vs {tuple(y.shape)}')
            if name in ('R51-IdentityTemporal','R51-Combined'):
                # 更改未来时刻的输入，早期hidden不得改变；检验TemporalMixer因果性
                z=m.encoder(x)
                prev=m.temporal(z)
                z2=z.clone();z2[:,90:]+=torch.randn_like(z2[:,90:])*0.5
                other=m.temporal(z2)
                causal_error=float((prev[:,:90]-other[:,:90]).abs().max())
                if causal_error>1e-5:raise RuntimeError(f'{name} temporal convolution leakage={causal_error}')
        m.train();p=m(x);loss=model_loss(p.float(),y.float(),fd,b,name)
        if not torch.isfinite(loss):raise RuntimeError(name+' nan loss')
        loss.backward()
        g=sum(v.grad is not None and torch.isfinite(v.grad).all() for v in m.parameters() if v.requires_grad)
        if g<3:raise RuntimeError(name+' insufficient gradients')
        smoke.append({'模型':name,'分支':b,'参数量':nparams(m),'loss':float(loss.detach()),'梯度层数':g,'结果':'PASS'})
        del m,x,y,p,loss;torch.cuda.empty_cache()
pd.DataFrame(smoke).to_csv(DATA/'R51_01_模型形状因果梯度预检.csv',index=False,encoding='utf-8-sig')
print('[R51 SHAPE+CAUSAL+GRAD PRECHECK] ALL PASS');display(pd.DataFrame(smoke))
del fd;gc.collect()

In [ ]:
# ===== 11. 内层早停与独立QA时间段：不能拿QA来早停或改训练损失 =====
def make_pilot_fold(f):
    year=f['year'];inner_year=year-2; qa_year=year-1
    qa_start=pd.Timestamp(qa_year,1,1)
    early_start=pd.Timestamp(inner_year,1,1)
    train_cut=early_start-pd.Timedelta(days=cfg.PURGE_DAYS)
    train=ISS[cal[ISS+L]<train_cut]
    # early-stop inner year issue，最后标签必须在QA起报之前
    early=ISS[(cal[ISS]>=early_start)&(cal[ISS]<qa_start)&(cal[ISS+L]<qa_start-pd.Timedelta(days=cfg.PURGE_DAYS))]
    qa=f['val']
    if len(train)<cfg.MIN_TRAIN or len(early)<cfg.MIN_VAL or len(qa)<cfg.MIN_VAL:
        raise RuntimeError(f'Pilot split {year} insufficient train={len(train)} early={len(early)} QA={len(qa)}')
    if not (cal[train+L].max()<cal[early].min()-pd.Timedelta(days=cfg.PURGE_DAYS) and
            cal[early+L].max()<cal[qa].min()-pd.Timedelta(days=cfg.PURGE_DAYS)):
        raise RuntimeError('Pilot temporal leakage')
    return {'fold':500+f['fold'],'year':f['year'],'train':train,'val':early,'qa':qa,'train_cut':train_cut}

PILOTS=[make_pilot_fold(next(f for f in FOLDS if f['year']==yr)) for yr in cfg.PILOT_TEST_YEARS]
pp=[]
for f in PILOTS:
    pp.append({'对应测试年份':f['year'],'训练样本':len(f['train']),'内层早停样本':len(f['val']),
               '独立机制资格样本':len(f['qa']), '训练最晚标签':str(cal[f['train']+L].max().date()),
               '早停最晚标签':str(cal[f['val']+L].max().date()),'QA最早起报':str(cal[f['qa']].min().date())})
pd.DataFrame(pp).to_csv(DATA/'R51_02_机制资格实验严格时间划分.csv',index=False,encoding='utf-8-sig')
print('[PILOT STRICT TEMPORAL QA] PASS');display(pd.DataFrame(pp))

In [ ]:
# ===== 12. 资格实验：先测4个网络×2fold×2seeds；不触碰test labels =====
PILOT_MODELS=['Matched-LSTM','EOF-TCN-LSTM','R51-IdentityTemporal','R51-PersistenceDelta','R51-Combined']
qa_csv=DATA/'R51_03_机制资格QA逐样本误差.csv'
qa_cache=OUT/'资格实验预测缓存';qa_cache.mkdir(parents=True,exist_ok=True)
rows=[]
for f in PILOTS:
    print('\n[PILOT]',f['year'],'QA year',f['year']-1)
    fd=make_fold_data(f)
    for name in PILOT_MODELS:
        for seed in cfg.PILOT_SEEDS:
            file=qa_cache/f'{f["year"]}_{name}_seed{seed}.npz'
            if file.exists() and not cfg.FORCE_TRAIN:
                with np.load(file) as z: a=z['b1'];b=z['b2']; ix=z['issues']
                if not np.array_equal(ix,f['qa']):raise RuntimeError(f'{file} QA indices mismatch')
            else:
                a,b=get_predictions(name,f,fd,seed,OUT/'资格实验权重',cfg.PILOT_EPOCHS,cfg.PILOT_PATIENCE)
                np.savez_compressed(file,b1=a,b2=b,issues=f['qa'])
            info=record_errors(f,name,seed,f['qa'],a,b)
            info['QA年份']=f['year']-1
            rows.append(info)
            pd.concat(rows,ignore_index=True).to_csv(qa_csv,index=False,encoding='utf-8-sig')
    del fd;gc.collect();torch.cuda.empty_cache()
QA=pd.concat(rows,ignore_index=True)
print('[PILOT DONE]',len(QA),'rows')

In [ ]:
# ===== 13. 按独立QA物理路径RMSE选机制，并实施固定资格门槛 =====
def aggregate_rmse(df):
    run=df.groupby(['QA年份','模型','种子','提前期_天'],as_index=False).agg(
        RMSE_km=('横向路径误差MSE_km2',lambda a:float(np.sqrt(np.mean(a)))))
    yr=run.groupby(['QA年份','模型','提前期_天'],as_index=False)['RMSE_km'].mean()
    agg=yr.groupby(['模型','提前期_天'],as_index=False)['RMSE_km'].mean()
    return run,yr,agg
runqa,yearqa,qa_mean=aggregate_rmse(QA)
qa_mean.to_csv(DATA/'R51_04_机制资格QA总体误差.csv',index=False,encoding='utf-8-sig')
ref=qa_mean[qa_mean['模型']=='Matched-LSTM'].set_index('提前期_天')['RMSE_km']
orig=qa_mean[qa_mean['模型']=='EOF-TCN-LSTM'].set_index('提前期_天')['RMSE_km']
checks=[]
for name in R51_NAMES:
    z=qa_mean[qa_mean['模型']==name].set_index('提前期_天')['RMSE_km']
    gain=float((ref-z).mean())
    gain_vs_orig=float((orig-z).mean())
    max_reg=float((z-ref).max())
    # 提前期等权，拒绝通过单个提前期的极端取巧改善掩盖显著回退
    ok=(gain>=cfg.GATE_MIN_GAIN_KM and gain_vs_orig>0 and max_reg<=cfg.GATE_MAX_LEAD_REGRESSION_KM)
    checks.append({'候选':name,'相对MatchedLSTM改善_km':gain,'相对原ETL改善_km':gain_vs_orig,
                   '最大单提前期退化_km':max_reg,'通过机制资格':bool(ok)})
check_df=pd.DataFrame(checks).sort_values('相对MatchedLSTM改善_km',ascending=False)
check_df.to_csv(DATA/'R51_05_机制资格硬门槛.csv',index=False,encoding='utf-8-sig')
qualified=check_df[check_df['通过机制资格']]
SELECTED_R5=qualified.iloc[0]['候选'] if len(qualified) else None
qual_record={'qualified':SELECTED_R5 is not None,'selected':SELECTED_R5,
             'gate':{'minimum_mean_gain_km':cfg.GATE_MIN_GAIN_KM,
                     'maximum_single_lead_regression_km':cfg.GATE_MAX_LEAD_REGRESSION_KM},
             'QA_years':[int(y-1) for y in cfg.PILOT_TEST_YEARS]}
(REP/'R51_机制资格判定.json').write_text(json.dumps(qual_record,ensure_ascii=False,indent=2),encoding='utf-8')
display(qa_mean);display(check_df)
print('R51 EXPLORATORY QUALIFICATION:',SELECTED_R5 if SELECTED_R5 else 'FAILED: Do not run full training')

In [ ]:

# ===== 14. 额外时间段机制确认（仅在第一阶段PASS时运行，不触碰正式test标签） =====
CONFIRM_OK=False
confirm_rows=[]
confirm_gate={"exploratory_qualified":SELECTED_R5 is not None,"confirmation_pass":False,
              "confirmation_qa_years":[2012,2020],"selected":SELECTED_R5}
if SELECTED_R5 is None:
    print('[R51 STOP] 2008/2016研发QA未通过，跳过额外确认；已保存所有失败结果。')
elif not cfg.RUN_CONFIRMATION:
    print('[R51 NOTE] 确认阶段被禁用；不能视为通过跨年机制验证。')
else:
    confirm_folds=[make_pilot_fold(next(f for f in FOLDS if f['year']==y)) for y in cfg.CONFIRM_PILOT_YEARS]
    confirm_models=['Matched-LSTM','EOF-TCN-LSTM',SELECTED_R5]
    confirm_cache=OUT/'额外年份确认缓存'
    confirm_cache.mkdir(parents=True,exist_ok=True)
    for f in confirm_folds:
        fd=make_fold_data(f)
        for name in confirm_models:
            for seed in cfg.PILOT_SEEDS:
                file=confirm_cache/f'{f["year"]}_{name}_seed{seed}.npz'
                if file.exists() and not cfg.FORCE_TRAIN:
                    with np.load(file) as z:a=z['b1'];b=z['b2'];ix=z['issues']
                    if not np.array_equal(ix,f['qa']):raise RuntimeError('confirmation cache indices mismatch')
                else:
                    aa,bb=get_predictions(name,f,fd,seed,OUT/'额外年份确认权重',cfg.PILOT_EPOCHS,cfg.PILOT_PATIENCE)
                    a,b=aa,bb
                    np.savez_compressed(file,b1=a,b2=b,issues=f['qa'])
                met=record_errors(f,name,seed,f['qa'],a,b)
                met['QA年份']=f['year']-1
                confirm_rows.append(met)
                pd.concat(confirm_rows,ignore_index=True).to_csv(
                    DATA/'R51_06_额外年份QA逐样本误差.csv',index=False,encoding='utf-8-sig')
        del fd
        gc.collect();torch.cuda.empty_cache()
    cqa=pd.concat(confirm_rows,ignore_index=True)
    _,cyr,cmean=aggregate_rmse(cqa)
    cmean.to_csv(DATA/'R51_07_额外年份QA总体误差.csv',index=False,encoding='utf-8-sig')
    cref=cmean[cmean['模型']=='Matched-LSTM'].set_index('提前期_天')['RMSE_km']
    corig=cmean[cmean['模型']=='EOF-TCN-LSTM'].set_index('提前期_天')['RMSE_km']
    cz=cmean[cmean['模型']==SELECTED_R5].set_index('提前期_天')['RMSE_km']
    cgain=float((cref-cz).mean())
    cgain_orig=float((corig-cz).mean())
    cmaxreg=float((cz-cref).max())
    CONFIRM_OK=bool(cgain>=cfg.GATE_MIN_GAIN_KM and cgain_orig>0 and cmaxreg<=cfg.GATE_MAX_LEAD_REGRESSION_KM)
    confirm_gate.update({"confirmation_pass":CONFIRM_OK,"gain_vs_matched_lstm_km":cgain,
                        "gain_vs_orig_etl_km":cgain_orig,"max_single_lead_regression_km":cmaxreg})
    display(cmean)
    print('[R51 CONFIRM] PASS' if CONFIRM_OK else '[R51 CONFIRM] FAIL',confirm_gate)
(REP/'R51_额外年份机制确认判定.json').write_text(
    json.dumps(confirm_gate,ensure_ascii=False,indent=2),encoding='utf-8')


In [ ]:

# ===== 15. 机制QA图与审稿交接；绝不在本Notebook中正式长训 =====
colors={'Matched-LSTM':'#80ABC2','EOF-TCN-LSTM':'#557F95',
        'R51-IdentityTemporal':'#427DA8','R51-PersistenceDelta':'#C88A68','R51-Combined':'#356C83'}
zh={'Matched-LSTM':'同参数量LSTM','EOF-TCN-LSTM':'原EOF-TCN-LSTM',
    'R51-IdentityTemporal':'近恒等时序卷积','R51-PersistenceDelta':'EOF状态增量预测',
    'R51-Combined':'时序卷积与增量预测'}

def save_qa_fig(df,tag,title_zh,title_en):
    for lang,folder,stem in [('zh',FZH,f'{tag}_中文'),('en',FEN,f'{tag}_English')]:
        plt.rcdefaults()
        matplotlib.rcParams['font.family']=[CJK_NAME,'DejaVu Sans'] if lang=='zh' else ['DejaVu Serif']
        matplotlib.rcParams.update({'axes.grid':True,'grid.color':'#DDDDDD','grid.linestyle':':',
            'xtick.direction':'in','ytick.direction':'in','axes.unicode_minus':False,'pdf.fonttype':42})
        fig,ax=plt.subplots(figsize=(9.0,5.4))
        for name in df['模型'].unique():
            q=df[df['模型']==name].sort_values('提前期_天')
            ax.plot(q['提前期_天'],q['RMSE_km'],color=colors.get(name,'#777777'),
                    marker='o',markerfacecolor='white',label=zh.get(name,name) if lang=='zh' else name)
        ax.set_xticks(list(cfg.LEADS))
        ax.set_xlabel('预测提前期（天）' if lang=='zh' else 'Forecast lead (days)')
        ax.set_ylabel('路径RMSE（km）' if lang=='zh' else 'Axis RMSE (km)')
        ax.set_title(title_zh if lang=='zh' else title_en)
        ax.legend(ncol=2,frameon=False);fig.tight_layout()
        fig.savefig(folder/(stem+'.png'),dpi=cfg.FIG_DPI,bbox_inches='tight')
        fig.savefig(folder/(stem+'.pdf'),bbox_inches='tight');plt.close(fig)

save_qa_fig(qa_mean,'R51_研发机制资格对比','R5.1机制资格对比：2008与2016年','R5.1 exploratory mechanism QA: 2008 and 2016')
if confirm_rows:
    save_qa_fig(cmean,'R51_额外时间段确认','R5.1额外确认：2012与2020年','R5.1 additional temporal confirmation: 2012 and 2020')

report={
 'experiment_version':'R5.1-identity-v1',
 'original_r5_failure_confirmed':True,
 'exploratory_QA_years':[2008,2016],
 'exploratory_gate':qual_record,
 'additional_QA_years':[2012,2020],
 'confirmation_gate':confirm_gate,
 'recommended_to_proceed_formal_training':bool(SELECTED_R5 is not None and CONFIRM_OK),
 'important_note':'旧R4测试结果已被查看；未来多年份正式结果必须完整披露，不能称为真正盲测。'
}
(REP/'R51_最终诊断交接.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('\n[R51 FINISHED] 没有运行正式长训。')
print('第一阶段资格:',SELECTED_R5)
print('第二阶段确认:',CONFIRM_OK)
print('输出目录:',OUT)
